In [2]:
import gzip
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

# paths (notebook is in notebooks/, so go up one level)
DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# verify the files we have
print("Files in data/raw:")
for f in sorted(DATA_DIR.glob("*.json.gz")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")

Files in data/raw:
  goodreads_book_authors.json.gz: 17.2 MB
  goodreads_book_genres_initial.json.gz: 23.1 MB
  goodreads_books.json.gz: 1986.7 MB


In [3]:
books_file = DATA_DIR / "goodreads_books.json.gz"

# read just the first record
with gzip.open(books_file, "rt", encoding="utf-8") as f:
    first_line = f.readline()
    sample = json.loads(first_line)

print("Keys in a book record:")
print(list(sample.keys()))
print(f"\nSample values (first 500 chars):")
print(json.dumps(sample, indent=2)[:500])

Keys in a book record:
['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']

Sample values (first 500 chars):
{
  "isbn": "0312853122",
  "text_reviews_count": "1",
  "series": [],
  "country_code": "US",
  "language_code": "",
  "popular_shelves": [
    {
      "count": "3",
      "name": "to-read"
    },
    {
      "count": "1",
      "name": "p"
    },
    {
      "count": "1",
      "name": "collection"
    },
    {
      "count": "1",
      "name": "w-c-fields"
    },
    {
      "count": "1",
      "name": "biography"
    }
  ],
  "asin": "",
  "is_ebook": "false",
  "average_rating": "4.00",
  "


In [4]:
genres_file = DATA_DIR / "goodreads_book_genres_initial.json.gz"

print("Loading genre tags...")
book_genres = {}  # book_id -> dict of genre: count
with gzip.open(genres_file, "rt", encoding="utf-8") as f:
    for line in tqdm(f):
        rec = json.loads(line)
        book_genres[rec["book_id"]] = rec.get("genres", {})

print(f"\nGenre tags loaded for {len(book_genres):,} books")

# peek at structure
sample_keys = list(book_genres.keys())[:5]
for k in sample_keys:
    print(f"  {k}: {book_genres[k]}")

Loading genre tags...


2360655it [00:05, 464007.00it/s]


Genre tags loaded for 2,360,655 books
  5333265: {'history, historical fiction, biography': 1}
  1333909: {'fiction': 219, 'history, historical fiction, biography': 5}
  7327624: {'fantasy, paranormal': 31, 'fiction': 8, 'mystery, thriller, crime': 1, 'poetry': 1}
  6066819: {'fiction': 555, 'romance': 23, 'mystery, thriller, crime': 10}
  287140: {'non-fiction': 3}


In [5]:
def extract_book(book, genres_lookup):
    """Pull just the fields we need, with proper type conversion."""
    def safe_int(val):
        try:
            return int(val) if val not in ("", None) else None
        except (ValueError, TypeError):
            return None
    
    def safe_float(val):
        try:
            return float(val) if val not in ("", None) else None
        except (ValueError, TypeError):
            return None
    
    author_ids = [a.get("author_id", "") for a in book.get("authors", []) if a.get("author_id")]
    book_id = book.get("book_id", "")
    
    # genres from the genres file - keep top 3 by count
    genre_dict = genres_lookup.get(book_id, {})
    top_genres = sorted(genre_dict.items(), key=lambda x: -x[1])[:3]
    genres = [g[0] for g in top_genres]
    
    return {
        "book_id": book_id,
        "work_id": book.get("work_id", ""),
        "title": book.get("title", "").strip(),
        "description": book.get("description", "").strip(),
        "author_ids": author_ids,
        "genres": genres,
        "language_code": book.get("language_code", ""),
        "num_pages": safe_int(book.get("num_pages")),
        "publication_year": safe_int(book.get("publication_year")),
        "average_rating": safe_float(book.get("average_rating")),
        "ratings_count": safe_int(book.get("ratings_count")) or 0,
        "text_reviews_count": safe_int(book.get("text_reviews_count")) or 0,
        "is_ebook": book.get("is_ebook", False),
        "image_url": book.get("image_url", ""),
        "url": book.get("url", ""),
    }


def passes_filters(book):
    """v0 filters: English, has description >= 100 chars, has >= 10 ratings."""
    if book.get("language_code", "") not in ("eng", "en-US", "en-GB", "en-CA", "en"):
        return False
    
    desc = book.get("description", "")
    if not desc or len(desc) < 100:
        return False
    
    if not book.get("title", "").strip():
        return False
    
    try:
        ratings = int(book.get("ratings_count", "0"))
    except (ValueError, TypeError):
        ratings = 0
    if ratings < 10:
        return False
    
    return True

print("Functions defined.")

Functions defined.


In [6]:
print("Filtering full corpus (~2.36M books)...")
all_books = []
total_seen = 0

with gzip.open(books_file, "rt", encoding="utf-8") as f:
    for line in tqdm(f, total=2360655):
        book = json.loads(line)
        total_seen += 1
        if passes_filters(book):
            all_books.append(extract_book(book, book_genres))

print(f"\nProcessed:      {total_seen:,} books")
print(f"Passed filters: {len(all_books):,}")
print(f"Filtered out:   {total_seen - len(all_books):,} ({100*(total_seen - len(all_books))/total_seen:.1f}%)")

Filtering full corpus (~2.36M books)...


100%|██████████| 2360655/2360655 [01:33<00:00, 25333.49it/s]


Processed:      2,360,655 books
Passed filters: 586,510
Filtered out:   1,774,145 (75.2%)


In [7]:
df = pd.DataFrame(all_books)
print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

# free the big list now that it's in a DataFrame (helps RAM)
del all_books

# save
output_path = PROCESSED_DIR / "books_v0.parquet"
df.to_parquet(output_path, index=False)
print(f"\nSaved to: {output_path}")
print(f"File size: {output_path.stat().st_size / (1024**2):.1f} MB")

DataFrame shape: (586510, 15)
Memory usage: 772.6 MB

Saved to: ..\data\processed\books_v0.parquet
File size: 389.9 MB


In [8]:
# top books by ratings - should be recognizable across many genres now
print("Top 15 books by ratings_count:")
print(df.nlargest(15, "ratings_count")[["title", "genres", "ratings_count", "average_rating"]].to_string())

# basic stats
print(f"\n\nCorpus stats:")
print(f"  Total books:        {len(df):,}")
print(f"  Unique book_ids:    {df['book_id'].nunique():,}")
print(f"  Have page count:    {df['num_pages'].notna().sum():,}")
print(f"  Have pub year:      {df['publication_year'].notna().sum():,}")
print(f"  Mean description len: {df['description'].str.len().mean():.0f} chars")

# publication year range (filter out garbage years)
years = df['publication_year'].dropna()
years = years[(years > 1800) & (years <= 2017)]
print(f"  Pub year range:     {int(years.min())} - {int(years.max())}")
print(f"  Median pub year:    {int(years.median())}")

Top 15 books by ratings_count:
                                                              title                                                                       genres  ratings_count  average_rating
127927                      The Hunger Games (The Hunger Games, #1)                                  [young-adult, fiction, fantasy, paranormal]        4899965            4.34
394653     Harry Potter and the Sorcerer's Stone (Harry Potter, #1)                                  [fantasy, paranormal, young-adult, fiction]        4765497            4.45
145260                                      Twilight (Twilight, #1)                                  [fantasy, paranormal, young-adult, fiction]        3941381            3.57
355059                                        To Kill a Mockingbird               [fiction, history, historical fiction, biography, young-adult]        3255518            4.26
207866                                             The Great Gatsby                   [fi

In [9]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
assert device == "cuda", "GPU not available! Stop and check torch install."

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"Model loaded. Output dim: {model.get_sentence_embedding_dimension()}")

# quick test
test = model.encode(["A test sentence about wizards and magic."], 
                     convert_to_numpy=True, normalize_embeddings=True)
print(f"Test embedding shape: {test.shape}")

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\aryas\AppData\Local\Temp\ipykernel_20520\2107367171.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Output dim: {model.get_sentence_embedding_dimension()}")


Model loaded. Output dim: 384
Test embedding shape: (1, 384)


In [10]:
# reload corpus (ensures we embed exactly what's saved)
df = pd.read_parquet(PROCESSED_DIR / "books_v0.parquet")
descriptions = df["description"].tolist()
print(f"Embedding {len(descriptions):,} descriptions...")

embeddings = model.encode(
    descriptions,
    batch_size=128,          # bumped from 64 - your earlier run showed GPU wasn't saturated
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Memory: {embeddings.nbytes / (1024**2):.1f} MB")

# save
embeddings_path = PROCESSED_DIR / "embeddings_minilm_v0.npy"
np.save(embeddings_path, embeddings)
print(f"Saved to: {embeddings_path}")

Embedding 586,510 descriptions...


Batches:   0%|          | 0/4583 [00:00<?, ?it/s]


Embeddings shape: (586510, 384)
Memory: 859.1 MB
Saved to: ..\data\processed\embeddings_minilm_v0.npy


In [11]:
from rapidfuzz import fuzz, process

print("Loading author names...")
author_id_to_name = {}
authors_file = DATA_DIR / "goodreads_book_authors.json.gz"
with gzip.open(authors_file, "rt", encoding="utf-8") as f:
    for line in tqdm(f):
        a = json.loads(line)
        author_id_to_name[a["author_id"]] = a["name"]

print(f"Authors loaded: {len(author_id_to_name):,}")

def get_primary_author(author_ids):
    if author_ids is None or len(author_ids) == 0:
        return ""
    return author_id_to_name.get(author_ids[0], "")

df["primary_author"] = df["author_ids"].apply(get_primary_author)
print(f"Authors resolved: {(df['primary_author'] != '').sum():,} / {len(df):,}")

Loading author names...


829529it [00:01, 455638.65it/s]


Authors loaded: 829,529
Authors resolved: 586,510 / 586,510


In [12]:
ratings_df = pd.read_csv("../data/my_ratings.csv")
for col in ["title", "author", "notes"]:
    if col in ratings_df.columns:
        ratings_df[col] = ratings_df[col].astype(str).str.strip()

print(f"Loaded {len(ratings_df)} ratings\n")


def find_best_match(query_title, query_author, corpus_df, title_threshold=80, author_threshold=70):
    """Find best matching book in corpus."""
    titles = corpus_df["title"].fillna("").tolist()
    
    title_matches = process.extract(query_title, titles, scorer=fuzz.WRatio, limit=20)
    if not title_matches or title_matches[0][1] < title_threshold:
        return None
    
    best = None
    best_score = 0
    for matched_title, title_score, idx in title_matches:
        candidate = corpus_df.iloc[idx]
        candidate_author = candidate["primary_author"]
        
        author_score = fuzz.WRatio(query_author, candidate_author) if query_author else 100
        combined = title_score * 0.6 + author_score * 0.4
        
        if combined > best_score and author_score >= author_threshold:
            best_score = combined
            best = {
                "book_id": candidate["book_id"],
                "matched_title": matched_title,
                "matched_author": candidate_author,
                "title_score": title_score,
                "author_score": author_score,
                "combined_score": combined,
                "corpus_index": idx,
            }
    return best


matched_ratings = []
unmatched = []

for _, row in ratings_df.iterrows():
    result = find_best_match(row["title"], row["author"], df)
    if result:
        result["my_title"] = row["title"]
        result["my_author"] = row["author"]
        result["rating"] = row["rating"]
        matched_ratings.append(result)
        print(f"✓ {row['title'][:35]:35s} -> {result['matched_title'][:45]:45s} by {result['matched_author'][:20]:20s} (T:{result['title_score']:.0f} A:{result['author_score']:.0f})")
    else:
        unmatched.append({"title": row["title"], "author": row["author"], "rating": row["rating"]})
        print(f"✗ {row['title'][:35]:35s} -> NO MATCH")

print(f"\nMatched: {len(matched_ratings)} / {len(ratings_df)}")
print(f"Unmatched: {len(unmatched)}")

Loaded 28 ratings

✓ Dark Matter                         -> Dark Matter                                   by Blake Crouch         (T:100 A:100)
✗ The Eyes Are The Best Part          -> NO MATCH
✓ One Hundred Years of Solitude       -> One Hundred Years of Solitude                 by Gabriel Garcia Marqu (T:100 A:91)
✓ The Lovecraft Compendium            -> The Lovecraft Compendium                      by H.P. Lovecraft       (T:100 A:100)
✓ The Palace of Illusions             -> The Palace of Illusions                       by Chitra Banerjee Diva (T:100 A:100)
✓ The Alchemist                       -> The Alchemist                                 by Paulo Coelho         (T:100 A:100)
✗ The Poppy War                       -> NO MATCH
✓ Divergent                           -> Divergent                                     by Veronica Roth        (T:100 A:100)
✓ Thank You For Smoking               -> Thank You For Smoking                         by Christopher Buckley  (T:100 A:100)
✗ Yello

In [13]:
import faiss

# load embeddings (in case kernel restart, we want this to work standalone)
embeddings = np.load(PROCESSED_DIR / "embeddings_minilm_v0.npy").astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

# build a flat index using inner product (= cosine similarity since vectors are normalized)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal:,}")

# save it
faiss_path = PROCESSED_DIR / "faiss_minilm_v0.index"
faiss.write_index(index, str(faiss_path))
print(f"Saved to: {faiss_path}")

Embeddings shape: (586510, 384)
FAISS index built. Total vectors: 586,510
Saved to: ..\data\processed\faiss_minilm_v0.index


In [14]:
def build_taste_vector(matched_ratings, embeddings):
    """
    Compute weighted average of rated-book embeddings.
    Weight = (rating - 3), so 5★ pulls strongly, 1★ pushes away, 3★ ignored.
    """
    vectors = []
    weights = []
    for r in matched_ratings:
        idx = r["corpus_index"]
        weight = r["rating"] - 3.0
        if weight == 0:  # skip neutral ratings - they contribute nothing
            continue
        vectors.append(embeddings[idx])
        weights.append(weight)
    
    if not vectors:
        return None
    
    vectors = np.array(vectors)
    weights = np.array(weights).reshape(-1, 1)
    
    # weighted average
    taste_vector = (vectors * weights).sum(axis=0) / np.abs(weights).sum()
    
    # normalize so we can use inner product = cosine similarity
    taste_vector = taste_vector / np.linalg.norm(taste_vector)
    return taste_vector.astype("float32").reshape(1, -1)


def recommend(matched_ratings, df, embeddings, index, n=20, exclude_rated=True):
    """Find top-n books most similar to user's taste vector."""
    taste_vector = build_taste_vector(matched_ratings, embeddings)
    if taste_vector is None:
        return None
    
    # query more than n in case we filter some out
    k = n + len(matched_ratings) + 50
    similarities, indices = index.search(taste_vector, k)
    similarities = similarities[0]
    indices = indices[0]
    
    rated_book_ids = {r["book_id"] for r in matched_ratings}
    
    results = []
    for sim, idx in zip(similarities, indices):
        book = df.iloc[idx]
        if exclude_rated and book["book_id"] in rated_book_ids:
            continue
        results.append({
            "title": book["title"],
            "author": book["primary_author"],
            "genres": list(book["genres"]) if book["genres"] is not None else [],
            "year": book["publication_year"],
            "avg_rating": book["average_rating"],
            "ratings_count": book["ratings_count"],
            "similarity": float(sim),
            "corpus_index": int(idx),
        })
        if len(results) >= n:
            break
    
    return pd.DataFrame(results)


# show the user their taste vector breakdown first
print("=== Your rated books (matched to corpus) ===")
for r in sorted(matched_ratings, key=lambda x: -x["rating"]):
    weight = r["rating"] - 3.0
    direction = "→" if weight > 0 else ("←" if weight < 0 else "·")
    print(f"  {r['rating']:.1f} {direction} {r['matched_title'][:50]:50s} (weight: {weight:+.1f})")

print("\n=== Top 20 recommendations ===")
recs = recommend(matched_ratings, df, embeddings, index, n=20)
print(recs[["title", "author", "genres", "year", "similarity"]].to_string())

=== Your rated books (matched to corpus) ===
  4.5 → The Palace of Illusions                            (weight: +1.5)
  4.5 → The Hitchhiker's Guide to the Galaxy               (weight: +1.5)
  4.5 → A Walk in the Woods                                (weight: +1.5)
  4.0 → Thank You For Smoking                              (weight: +1.0)
  4.0 → Tony Bourdain Boxset - Kitchen Confidential & Medi (weight: +1.0)
  4.0 → The Storied Life of A.J. Fikry                     (weight: +1.0)
  4.0 → The Namesake                                       (weight: +1.0)
  4.0 → Lamb                                               (weight: +1.0)
  3.5 → Dark Matter                                        (weight: +0.5)
  3.5 → Geek Love                                          (weight: +0.5)
  3.5 → Carrie                                             (weight: +0.5)
  3.5 → The Hotel New Hampshire                            (weight: +0.5)
  3.5 → The Maltese Falcon                                 (weight:

In [15]:
# dedupe by work_id, keeping the edition with the most ratings
print(f"Before dedup: {len(df):,} books")

# some work_ids are empty - treat those as unique (keep them all)
df_with_work = df[df["work_id"] != ""].copy()
df_no_work = df[df["work_id"] == ""].copy()

# for books with a work_id, keep the most-rated edition
idx_to_keep = df_with_work.groupby("work_id")["ratings_count"].idxmax()
df_deduped = pd.concat([df_with_work.loc[idx_to_keep], df_no_work]).sort_index()

print(f"After dedup:  {len(df_deduped):,} books")
print(f"Removed {len(df) - len(df_deduped):,} duplicate editions")

# we need embeddings aligned to the deduped df.
# df_deduped keeps original indices, so we select those rows from embeddings
keep_positions = df_deduped.index.to_numpy()
embeddings_deduped = embeddings[keep_positions]

# reset index so positions align with the new embeddings array
df_deduped = df_deduped.reset_index(drop=True)

print(f"Deduped embeddings shape: {embeddings_deduped.shape}")

# rebuild FAISS on deduped set
index_deduped = faiss.IndexFlatIP(embeddings_deduped.shape[1])
index_deduped.add(embeddings_deduped)
print(f"Deduped FAISS index: {index_deduped.ntotal:,} vectors")

# save deduped artifacts
df_deduped.to_parquet(PROCESSED_DIR / "books_v0_deduped.parquet", index=False)
np.save(PROCESSED_DIR / "embeddings_minilm_v0_deduped.npy", embeddings_deduped)
faiss.write_index(index_deduped, str(PROCESSED_DIR / "faiss_minilm_v0_deduped.index"))
print("Saved deduped artifacts.")

Before dedup: 586,510 books
After dedup:  409,042 books
Removed 177,468 duplicate editions
Deduped embeddings shape: (409042, 384)
Deduped FAISS index: 409,042 vectors
Saved deduped artifacts.


In [16]:
# re-add primary_author to deduped df
df_deduped["primary_author"] = df_deduped["author_ids"].apply(get_primary_author)

# re-match ratings against deduped corpus
matched_ratings = []
unmatched = []
for _, row in ratings_df.iterrows():
    result = find_best_match(row["title"], row["author"], df_deduped)
    if result:
        result["my_title"] = row["title"]
        result["my_author"] = row["author"]
        result["rating"] = row["rating"]
        matched_ratings.append(result)
    else:
        unmatched.append({"title": row["title"], "author": row["author"]})

print(f"Re-matched: {len(matched_ratings)} / {len(ratings_df)}")

# collect the set of author names the user has already rated (for exclusion)
rated_authors = {r["matched_author"].lower() for r in matched_ratings if r["matched_author"]}
print(f"Authors you've already rated: {rated_authors}")

Re-matched: 20 / 28
Authors you've already rated: {'kurt vonnegut jr.', 'dashiell hammett', 'anthony bourdain', 'christopher moore', 'stephen king', 'gabrielle zevin', 'douglas adams', 'katherine dunn', 'christopher buckley', 'chitra banerjee divakaruni', 'john irving', 'veronica roth', 'gabriel garcia marquez', 'h.p. lovecraft', 'natsuo kirino', 'paulo coelho', 'blake crouch', 'jhumpa lahiri', 'bill bryson'}


In [ ]:
def recommend_v3(matched_ratings, df, embeddings, index, n=20,
                 exclude_rated=True, exclude_rated_authors=True,
                 min_ratings=1000, popularity_weight=0.15):
    """
    Rank by similarity blended with a popularity prior.
    min_ratings: hard floor - don't recommend books below this many ratings
    popularity_weight: how much to blend in log-popularity (0 = pure similarity)
    """
    taste_vector = build_taste_vector(matched_ratings, embeddings)
    if taste_vector is None:
        return None
    
    # query a large candidate pool since we filter aggressively
    k = 2000
    similarities, indices = index.search(taste_vector, k)
    similarities = similarities[0]
    indices = indices[0]
    
    rated_book_ids = {r["book_id"] for r in matched_ratings}
    rated_authors = {r["matched_author"].lower() for r in matched_ratings if r["matched_author"]}
    
    candidates = []
    for sim, idx in zip(similarities, indices):
        book = df.iloc[idx]
        if exclude_rated and book["book_id"] in rated_book_ids:
            continue
        if exclude_rated_authors and book["primary_author"].lower() in rated_authors:
            continue
        if book["ratings_count"] < min_ratings:
            continue
        candidates.append({
            "title": book["title"],
            "author": book["primary_author"],
            "genres": list(book["genres"]) if book["genres"] is not None else [],
            "year": book["publication_year"],
            "similarity": float(sim),
            "ratings_count": int(book["ratings_count"]),
            "avg_rating": float(book["average_rating"]) if pd.notna(book["average_rating"]) else 0,
        })
    
    if not candidates:
        return pd.DataFrame()
    
    cand_df = pd.DataFrame(candidates)
    
    # popularity score: log of ratings count, normalized to 0-1
    cand_df["log_pop"] = np.log10(cand_df["ratings_count"])
    cand_df["pop_norm"] = (cand_df["log_pop"] - cand_df["log_pop"].min()) / (cand_df["log_pop"].max() - cand_df["log_pop"].min() + 1e-9)
    
    # blended score
    cand_df["score"] = (1 - popularity_weight) * cand_df["similarity"] + popularity_weight * cand_df["pop_norm"]
    
    cand_df = cand_df.sort_values("score", ascending=False).head(n)
    return cand_df


print("=== Top 20 (popularity-weighted, min 1000 ratings) ===")
# more aggressive popularity bias
recs = recommend_v3(matched_ratings, df_deduped, embeddings_deduped, index_deduped, 
                    n=20, min_ratings=5000, popularity_weight=0.25)
print(recs[["title", "author", "genres", "similarity", "ratings_count"]].to_string())

=== Top 20 (popularity-weighted, min 1000 ratings) ===
                                                                                                title                  author                                                              genres    year  similarity  ratings_count
30                                                                                   The Great Gatsby     F. Scott Fitzgerald          [fiction, history, historical fiction, biography, romance]  2004.0    0.326372        2758812
0                                                       And Another Thing... (Hitchhiker's Guide, #6)             Eoin Colfer                     [fiction, fantasy, paranormal, comics, graphic]  2009.0    0.435593          20187
6                                                     Animal, Vegetable, Miracle: A Year of Food Life      Barbara Kingsolver      [non-fiction, history, historical fiction, biography, fiction]  2007.0    0.347250          84521
21                           